In [10]:
import pandas as pd
from joblib import load, dump
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


## 1. DIVISÃO EM TREINO E TESTE (Train-Test Split)

In [11]:
caminho_arquivo = "/home/sophia.ozorio/storage/python-ml/projeto_ML_python/dev/script/results/variantes-final-limpo-v1.pkl"

df_base = load(caminho_arquivo)
display(df_base.head())
print(df_base.shape)

,tipo_variante_frameshift,tipo_variante_missense,tipo_variante_nonsense,ancestralidade_AFR,ancestralidade_AMR,ancestralidade_Desconhecido,ancestralidade_EAS,ancestralidade_EUR,ancestralidade_SAS,gene_HTT,...,classificacao_clinica_Patogenica,classificacao_clinica_VUS,cadd_phred,conservacao_phylop,profundidade_leitura,em_dominio_funcional,historico_familiar,idade_inicio_sintomas,ataxia,gnomad_af_log
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.0,22.388,2.699,60.4,0,0,43.5,2,-3.731402
1,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,26.440,5.232,65.0,0,1,37.5,0,-6.380952
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,8.989,-1.480,80.0,0,0,51.8,1,-2.846937
3,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,0.0,1.0,18.539,-0.605,59.9,0,0,29.8,2,-2.915568
4,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,8.904,0.245,75.5,0,1,52.1,2,-3.330104


(1023, 24)


In [12]:
df_base = df_base.dropna(subset=['idade_inicio_sintomas'])

In [13]:
X = df_base.drop(columns=['ataxia', 'idade_inicio_sintomas'])

# Cria duas targets separadas
y_ataxia = df_base['ataxia']
y_idade = df_base['idade_inicio_sintomas']

# 1ª divisão: separa o teste (10%) do restante (treino+validação, 90%)
X_temp, X_test, y_ataxia_temp, y_ataxia_test, y_idade_temp, y_idade_test = train_test_split(
    X, y_ataxia, y_idade, test_size=0.10, random_state=42, stratify=y_ataxia
)

# 2ª divisão: do restante, separa treino (70% do total = 77,78% de 90%) e validação (20% do total = 22,22% de 90%)
X_train, X_valid, y_ataxia_train, y_ataxia_valid, y_idade_train, y_idade_valid = train_test_split(
    X_temp, y_ataxia_temp, y_idade_temp, test_size=(0.20 / 0.90), random_state=42, stratify=y_ataxia_temp
)

X_train_proc = X_train.copy()
X_valid_proc = X_valid.copy()
X_test_proc = X_test.copy()

print("-" * 10)
print(f"X_train: {X_train.shape} ({X_train.shape[0] / X.shape[0]:.0%})")
print(f"X_valid: {X_valid.shape} ({X_valid.shape[0] / X.shape[0]:.0%})")
print(f"X_test: {X_test.shape} ({X_test.shape[0] / X.shape[0]:.0%})")
print("-" * 10)
print(f"y_train: {y_ataxia_train.shape}")
print(f"y_valid: {y_ataxia_valid.shape}")
print(f"y_test: {y_ataxia_test.shape}")
print("-" * 10)
print(f"y_train: {y_idade_train.shape}")
print(f"y_valid: {y_idade_valid.shape}")
print(f"y_test: {y_idade_test.shape}")

----------
X_train: (715, 22) (70%)
X_valid: (205, 22) (20%)
X_test: (103, 22) (10%)
----------
y_train: (715,)
y_valid: (205,)
y_test: (103,)
----------
y_train: (715,)
y_valid: (205,)
y_test: (103,)


## 2. DEFINIÇÃO DAS COLUNAS

In [14]:
colunas_numericas = [
    'cadd_phred',
    'conservacao_phylop',
    'profundidade_leitura',
    'gnomad_af_log'
]

## 3. IMPUTAÇÃO  DADOS AUSENTES (FITTED NO TREINO, TRANSFORMADO EM TREINO/TESTE)

In [15]:
# Imputação de 'historico_familiar' pela moda
if 'historico_familiar' in X_train.columns:
    imputer_moda = SimpleImputer(strategy='most_frequent')
    X_train_proc[['historico_familiar']] = imputer_moda.fit_transform(X_train[['historico_familiar']])
    X_valid_proc[['historico_familiar']] = imputer_moda.transform(X_valid[['historico_familiar']])
    X_test_proc[['historico_familiar']] = imputer_moda.transform(X_test[['historico_familiar']])

# Imputação de 'cadd_phred' pela mediana
if 'cadd_phred' in X_train.columns:
    imputer_mediana = SimpleImputer(strategy='median')
    X_train_proc[['cadd_phred']] = imputer_mediana.fit_transform(X_train_proc[['cadd_phred']])
    X_valid_proc[['cadd_phred']] = imputer_mediana.transform(X_valid_proc[['cadd_phred']])
    X_test_proc[['cadd_phred']] = imputer_mediana.transform(X_test_proc[['cadd_phred']])


## 4. TRANSFORMAÇÃO DE ESCALA (StandardScaler)

In [16]:
scaler = StandardScaler()

# Filtra colunas numericas que realmente existem no Dataframe
cols_num_existentes = [c for c in colunas_numericas if c in X_train.columns]

# Fit no treino + transform no treino, teste e validação
X_train_proc[cols_num_existentes] = scaler.fit_transform(X_train_proc[cols_num_existentes])
X_valid_proc[cols_num_existentes] = scaler.transform(X_valid_proc[cols_num_existentes])
X_test_proc[cols_num_existentes] = scaler.transform(X_test_proc[cols_num_existentes])

## 5. VALIDAÇÃO E CONFIRMAÇÃO

In [17]:
print("--- Procesamento concluido com sucesso ---")
print(f"Valores nulos no treino: {X_train_proc[['historico_familiar', 'cadd_phred']].isna().sum().sum()}")
print(f"Valores nulos no teste: {X_test_proc[['historico_familiar', 'cadd_phred']].isna().sum().sum()}")

display(X_train_proc[cols_num_existentes].agg(['mean', 'std']).round(2))

--- Procesamento concluido com sucesso ---
Valores nulos no treino: 0
Valores nulos no teste: 0


,cadd_phred,conservacao_phylop,profundidade_leitura,gnomad_af_log
mean,-0.0,0.0,0.0,0.0
std,1.0,1.0,1.0,1.0


## 6. SALVAR

In [18]:
# Salvando os arquivos correspondentes na pasta de results (usando as variáveis processadas):
caminho_dir = "/home/sophia.ozorio/storage/python-ml/projeto_ML_python/dev/script/results/split_dataset/"

X_train_proc.to_csv(f"{caminho_dir}X_train_ready.csv", index=False)
X_valid_proc.to_csv(f"{caminho_dir}X_valid_ready.csv", index=False)
X_test_proc.to_csv(f"{caminho_dir}X_test_ready.csv", index=False)

# Salvando targets de ataxia
y_ataxia_train.to_csv(f"{caminho_dir}y_ataxia_train_ready.csv", index=False)
y_ataxia_valid.to_csv(f"{caminho_dir}y_ataxia_valid_ready.csv", index=False)
y_ataxia_test.to_csv(f"{caminho_dir}y_ataxia_test_ready.csv", index=False)

# Salvando targets de idade de início dos sintomas
y_idade_train.to_csv(f"{caminho_dir}y_idade_train_ready.csv", index=False)
y_idade_valid.to_csv(f"{caminho_dir}y_idade_valid_ready.csv", index=False)
y_idade_test.to_csv(f"{caminho_dir}y_idade_test_ready.csv", index=False)

print("Features e targets (Ataxia e Idade) salvas com sucesso!")

Features e targets (Ataxia e Idade) salvas com sucesso!
